# 6.1 闭源与开放权重模型 / Closed and open-weight models

通过 API 使用和下载权重自己运行有什么区别，怎样读懂一个模型的名称和规模，以及量化怎样让大模型装进有限的内存。

How using an API differs from downloading weights and running them, how to read the name and size of a model, and how quantization fits a large model into limited memory.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 从配置数出一个 Llama 式模型的参数量 / Count the parameters of a Llama-style model from its config

0.2 中数过 GPT-2 的参数量。这里对目前更常见的结构做同样的事，并计算不同精度下的内存占用。测试会把结果与 Hub 上登记的 SmolLM2-135M 参数量逐位比较，并判断一个 270 亿参数的模型在几种显存下能以哪些精度运行。

Module 0.2 counted the parameters of GPT-2. The same is done here for the architecture that is more common today, together with the memory at different precisions. The tests compare the result digit by digit with the parameter count of SmolLM2-135M registered on the Hub, and determine at which precisions a model with 27 billion parameters runs with several amounts of GPU memory.

In [ ]:
# Module 0.2 counted the parameters of GPT-2. Most current open-weight models use the Llama-style
# block instead: RMSNorm, SwiGLU, grouped-query attention, no bias terms, no position table (RoPE
# has no parameters). This exercise counts such a model from its config alone.


def llama_param_count(cfg):
    """Number of parameters of a Llama-style decoder, from a config dict with the keys
    vocab_size, hidden_size, intermediate_size, num_hidden_layers,
    num_attention_heads, num_key_value_heads, tie_word_embeddings.

    Per block (no biases anywhere):
      attention  q: d x d      k and v: d x d_kv each, with d_kv = (d / n_heads) * n_kv_heads      output: d x d
      SwiGLU     three matrices: d x m, d x m, m x d                (m = intermediate_size)
      RMSNorm    two weight vectors of length d
    Outside the blocks: the embedding V x d, one final RMSNorm (d), and an output matrix V x d
    only if tie_word_embeddings is False.
    """
    # TODO 1
    raise NotImplementedError


def model_bytes(n_params, bits_per_weight):
    """Memory of the weights in bytes."""
    # TODO 2
    raise NotImplementedError


def precisions_that_fit(n_params, memory_gb, overhead=1.2, options=(16, 8, 4)):
    """Which of the bit widths in `options` fit into memory_gb gigabytes (1 GB = 1e9 bytes)?

    Count the weights times `overhead`, a rough allowance for activations and the KV cache.
    Returns the fitting bit widths as a list, in the order given.
    """
    # TODO 3
    raise NotImplementedError

In [ ]:
# 检查 / checks
# 1. SmolLM2-135M, a small Llama-style model. The config values are copied from its config.json.
SMOL = {"vocab_size": 49152, "hidden_size": 576, "intermediate_size": 1536, "num_hidden_layers": 30,
        "num_attention_heads": 9, "num_key_value_heads": 3, "tie_word_embeddings": True}
n = llama_param_count(SMOL)
assert n == 134_515_008, f"the Hub reports 134,515,008 parameters for SmolLM2-135M, got {n:,}"

# 2. Untying the output matrix adds exactly one more V x d table.
assert llama_param_count({**SMOL, "tie_word_embeddings": False}) - n == 49152 * 576
# 3. Without grouped-query attention (as many key/value heads as query heads) k and v are full d x d matrices.
full = llama_param_count({**SMOL, "num_key_value_heads": 9})
assert full - n == 30 * 2 * 576 * (576 - 192)

# 4. Memory
assert model_bytes(1_000_000, 16) == 2_000_000 and model_bytes(1_000_000, 4) == 500_000
assert abs(model_bytes(n, 32) / 1e6 - 538.1) < 0.1                     # about 538 MB in float32

# 5. What fits where. Parameter counts as reported on the Hub on 2026-11-25.
QWEN_27B = 27_781_427_952          # Qwen/Qwen3.8-27B
assert precisions_that_fit(QWEN_27B, 24) == [4]                         # a 24 GB consumer GPU
assert precisions_that_fit(QWEN_27B, 48) == [8, 4]
assert precisions_that_fit(QWEN_27B, 80) == [16, 8, 4]
assert precisions_that_fit(QWEN_27B, 12) == []
assert precisions_that_fit(n, 1) == [16, 8, 4]

print(f"m61_params: all tests passed (SmolLM2-135M: {n:,} parameters)")
for bits in (32, 16, 8, 4):
    print(f"  Qwen3.8-27B at {bits:2d} bits per weight: {model_bytes(QWEN_27B, bits) / 1e9:6.1f} GB")

<details><summary><b>提示 1 / Hint 1</b></summary>

键和值的投影矩阵形状是 `d × d_kv`，其中 `d_kv = (d // n_heads) * n_kv_heads`。

The projection matrices of keys and values have shape `d × d_kv`, with `d_kv = (d // n_heads) * n_kv_heads`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

每个块的参数量是 `2*d*d + 2*d*d_kv + 3*d*m + 2*d`。

The parameters of one block are `2*d*d + 2*d*d_kv + 3*d*m + 2*d`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
def llama_param_count(cfg):
    V, d, m, L = cfg["vocab_size"], cfg["hidden_size"], cfg["intermediate_size"], cfg["num_hidden_layers"]
    d_head = d // cfg["num_attention_heads"]
    d_kv = d_head * cfg["num_key_value_heads"]              # grouped-query attention: fewer key/value heads
    attention = d * d + 2 * d * d_kv + d * d                # q, (k, v), output projection; no biases
    mlp = 3 * d * m                                         # SwiGLU: gate, up, down
    norms = 2 * d                                           # two RMSNorm weight vectors per block
    total = V * d + L * (attention + mlp + norms) + d       # embedding, blocks, final norm
    if not cfg["tie_word_embeddings"]:
        total += V * d                                      # a separate output matrix
    return total


def model_bytes(n_params, bits_per_weight):
    return n_params * bits_per_weight / 8


def precisions_that_fit(n_params, memory_gb, overhead=1.2, options=(16, 8, 4)):
    return [b for b in options if model_bytes(n_params, b) * overhead <= memory_gb * 1e9]
```

</details>

## 2. 8 位与 4 位量化对比 / 8-bit and 4-bit quantization compared

实现对称量化的两种做法，即整个张量共用一个缩放系数和每个通道各用一个，并把它们应用到 GPT-2 各层的权重矩阵上。测试会检查舍入误差的方差接近 $s^2/12$、单个异常值对两种做法的影响，并比较 GPT-2 在四种设置下的困惑度。运行后请查看打印出的五个困惑度。

Implement symmetric quantization in two ways, one scale for the whole tensor and one per channel, and apply them to the weight matrices of all GPT-2 layers. The tests check that the variance of the rounding error is close to $s^2/12$, examine the effect of a single outlier on the two ways, and compare the perplexity of GPT-2 under four settings. Look at the five printed perplexities.

需要联网下载模型。/ Downloads a model.

In [ ]:
import math
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Symmetric ("absmax") quantization stores every weight as a small integer q and one shared scale s,
# with w approximately equal to q * s. With b bits the integers run from -qmax to qmax, qmax = 2^(b-1) - 1.


def quantize_absmax(w, bits):
    """One scale for the whole tensor.

    scale = max|w| / qmax,   q = round(w / scale), clamped to [-qmax, qmax]
    Returns (q, scale): q is a float tensor holding integer values, scale a 0-dim tensor.
    """
    # TODO 1
    raise NotImplementedError


def quantize_per_channel(w, bits, channel_dim):
    """One scale per channel of a 2-D weight matrix.

    channel_dim = 1 means one scale for every column: the maximum is taken over the other dimension.
    Use w.abs().amax(dim=..., keepdim=True) so that the scales broadcast against w.
    Returns (q, scale) with scale of shape (1, n) or (n, 1).
    """
    # TODO 2
    raise NotImplementedError


def dequantize(q, scale):
    """Back to floating point."""
    # TODO 3
    raise NotImplementedError


@torch.no_grad()
def quantize_gpt2_(model, bits, per_channel):
    """Replace every weight matrix inside the Transformer blocks by its quantized-and-dequantized version, in place.

    Loop over model.named_parameters(); take those with p.dim() == 2 whose name starts with "transformer.h.".
    GPT-2 stores these matrices as (in, out), so the output channels are channel_dim = 1.
    Write the result back with p.copy_(...). Returns the number of weights that were quantized.
    The model still computes in float32; this simulates the loss of precision, not the memory saving.
    """
    # TODO 4
    raise NotImplementedError

In [ ]:
# 检查 / checks
import math
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(0)
w = torch.randn(200, 300)

# 1. Integers in range, and the largest weight is reproduced exactly.
for bits in (8, 4):
    q, s = quantize_absmax(w, bits)
    qmax = 2 ** (bits - 1) - 1
    assert torch.equal(q, q.round()) and q.abs().max() == qmax and q.unique().numel() <= 2 * qmax + 1
    back = dequantize(q, s)
    assert (back - w).abs().max() <= s / 2 + 1e-6, "rounding moves a weight by at most half a step"
    assert torch.isclose(back.abs().max(), w.abs().max())

# 2. The rounding error behaves like uniform noise with variance scale^2 / 12.
q, s = quantize_absmax(w, 8)
mse = ((dequantize(q, s) - w) ** 2).mean().item()
assert abs(mse / (s.item() ** 2 / 12) - 1) < 0.05, "mean squared error should be close to scale^2 / 12"
q4, s4 = quantize_absmax(w, 4)
ratio = ((dequantize(q4, s4) - w) ** 2).mean().item() / mse
assert 200 < ratio < 450, f"four bits fewer make the step about 18 times larger, the error about 330 times: {ratio:.0f}"

# 3. Per-channel scales protect the other channels from one outlier.
v = torch.randn(200, 300) * 0.02
v[5, 7] = 3.0                                                  # a single very large weight in column 7
qt, st = quantize_absmax(v, 4)
qc, sc = quantize_per_channel(v, 4, channel_dim=1)
assert sc.shape == (1, 300) and torch.equal(qc, qc.round()) and qc.abs().max() <= 7
err_t = ((dequantize(qt, st) - v) ** 2).mean().item()
err_c = ((dequantize(qc, sc) - v) ** 2).mean().item()
assert (qt != 0).sum() == 1, "with one scale for the whole tensor everything except the outlier rounds to zero"
assert err_c < err_t / 5
assert quantize_per_channel(v, 4, channel_dim=0)[1].shape == (200, 1)

# 4. GPT-2 at different bit widths
tok = AutoTokenizer.from_pretrained("gpt2")
TEXT = ("The city council met on Tuesday evening to discuss the new budget. Several residents spoke about the condition of local roads, "
        "and others asked for more funding for the public library. After two hours of debate, the council voted to approve the plan with minor changes. "
        "The mayor said that construction would begin in the spring and that the work was expected to take about six months. "
        "Local businesses welcomed the decision, although some owners worried that the roadworks would keep customers away during the summer.")
ids = tok(TEXT, return_tensors="pt").input_ids

def perplexity(bits=None, per_channel=False):
    model = AutoModelForCausalLM.from_pretrained("gpt2").eval()
    n = quantize_gpt2_(model, bits, per_channel) if bits else 0
    with torch.no_grad():
        logp = torch.log_softmax(model(ids).logits[0, :-1], -1)
    nll = -logp.gather(-1, ids[0, 1:, None]).mean().item()
    return math.exp(nll), n

base, _ = perplexity()
p8, n = perplexity(8)
assert n == 84_934_656, n                                      # the 48 weight matrices of the 12 blocks
p8c, _ = perplexity(8, True)
p4, _ = perplexity(4)
p4c, _ = perplexity(4, True)
assert p8 < base * 1.25 and p8c < base * 1.05, "8 bits should cost little"
assert p4 > base * 20, "4 bits with one scale per tensor should break the model"
assert p4c < base * 3 and p4c < p4 / 10, "per-channel scales should rescue most of it"

print(f"m61_quant: all tests passed ({n:,} weights quantized)")
print(f"  float32                      perplexity {base:8.2f}")
print(f"  8 bits, one scale per tensor  perplexity {p8:8.2f}")
print(f"  8 bits, one scale per channel perplexity {p8c:8.2f}")
print(f"  4 bits, one scale per tensor  perplexity {p4:8.2f}")
print(f"  4 bits, one scale per channel perplexity {p4c:8.2f}")

<details><summary><b>提示 1 / Hint 1</b></summary>

`qmax = 2 ** (bits - 1) - 1`，量化是 `torch.clamp(torch.round(w / scale), -qmax, qmax)`。

`qmax = 2 ** (bits - 1) - 1`, and quantization is `torch.clamp(torch.round(w / scale), -qmax, qmax)`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

按通道时，`channel_dim=1` 表示每一列一个缩放系数，最大值要沿另一个维度取，即 `w.abs().amax(dim=0, keepdim=True)`。

Per channel, `channel_dim=1` means one scale per column, and the maximum is taken along the other dimension, that is `w.abs().amax(dim=0, keepdim=True)`.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

写回权重用 `p.copy_(dequantize(q, s))`，整个函数已经在 `torch.no_grad()` 之下。

Write the weights back with `p.copy_(dequantize(q, s))`. The whole function already runs under `torch.no_grad()`.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer


def quantize_absmax(w, bits):
    qmax = 2 ** (bits - 1) - 1
    scale = w.abs().max() / qmax
    q = torch.clamp(torch.round(w / scale), -qmax, qmax)
    return q, scale


def quantize_per_channel(w, bits, channel_dim):
    qmax = 2 ** (bits - 1) - 1
    other = 1 - channel_dim
    scale = w.abs().amax(dim=other, keepdim=True) / qmax          # one scale per channel
    q = torch.clamp(torch.round(w / scale), -qmax, qmax)
    return q, scale


def dequantize(q, scale):
    return q * scale


@torch.no_grad()
def quantize_gpt2_(model, bits, per_channel):
    n = 0
    for name, p in model.named_parameters():
        if p.dim() == 2 and name.startswith("transformer.h."):
            if per_channel:
                q, s = quantize_per_channel(p, bits, channel_dim=1)   # GPT-2 stores weights as (in, out)
            else:
                q, s = quantize_absmax(p, bits)
            p.copy_(dequantize(q, s))
            n += p.numel()
    return n
```

</details>

## 3. 手算参数量、内存与量化 / Parameters, memory and quantization by hand

在纸上完成，可以使用计算器。
(a) 一个 Llama 式模型的配置是 $V = 1000$、$d = 512$、$m = 1024$、$L = 4$，8 个查询头、2 个键值头，输出矩阵与嵌入共用。它有多少参数？
(b) 一个 80.3 亿参数的模型，以 16 位、8 位保存各需多少内存？以平均每个权重 4.89 位的方式量化后是多少？
(c) 对权重 $(0.6,\ -0.2,\ 0.05,\ -1.4)$ 做 3 位对称量化。写出缩放系数、量化后的整数和还原后的值。哪些权重变成了 0？
(d) 一个混合专家模型的总参数量是 1 万亿，每个 token 激活 520 亿。以 8 位保存需要多少内存？每个 token 的计算量大致相当于多大的普通模型？

Work on paper. A calculator may be used.
(a) A Llama-style model has the config $V = 1000$, $d = 512$, $m = 1024$, $L = 4$, 8 query heads and 2 key-value heads, with the output matrix tied to the embedding. How many parameters does it have?
(b) How much memory does a model with 8.03 billion parameters need at 16 bits and at 8 bits? And after quantization to an average of 4.89 bits per weight?
(c) Quantize the weights $(0.6,\ -0.2,\ 0.05,\ -1.4)$ symmetrically to 3 bits. Give the scale, the integers and the restored values. Which weights become 0?
(d) A mixture-of-experts model has 1 trillion parameters in total and activates 52 billion per token. How much memory does it need at 8 bits? To an ordinary model of what size does its computation per token roughly correspond?

<details><summary><b>提示 1 / Hint 1</b></summary>

(a) $d_{kv} = (512 / 8) \times 2 = 128$。

(a) $d_{kv} = (512 / 8) \times 2 = 128$.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(c) 3 位时 $q_{\max} = 2^{2} - 1 = 3$。

(c) At 3 bits $q_{\max} = 2^{2} - 1 = 3$.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) 注意力为 $512^2 + 2 \times 512 \times 128 + 512^2 = 655{,}360$。SwiGLU 为 $3 \times 512 \times 1024 = 1{,}572{,}864$。归一化为 $1024$。每个块共 $2{,}229{,}248$，4 个块为 $8{,}916{,}992$。加上嵌入 $512{,}000$ 和最后的归一化 $512$，总计 $9{,}429{,}504$。

(b) 16 位为 $8.03 \times 10^9 \times 2 = 16.1$ GB。8 位为 $8.0$ GB。4.89 位为 $8.03 \times 10^9 \times 4.89 / 8 \approx 4.9$ GB。这与 llama.cpp 文档中 Llama-3.1-8B 的 Q4_K_M 版本的大小一致。

(c) $q_{\max} = 3$，$s = 1.4 / 3 \approx 0.467$。$w / s = (1.29,\ -0.43,\ 0.11,\ -3)$，四舍五入得 $(1,\ 0,\ 0,\ -3)$，还原为 $(0.467,\ 0,\ 0,\ -1.4)$。$-0.2$ 和 $0.05$ 的绝对值都小于 $s/2 \approx 0.233$，被舍入为 0。最大的权重被精确保留，较小的权重损失最重。

(d) $10^{12}$ 个参数每个 1 字节，共 $10^{12}$ 字节，即 1000 GB。每个 token 的计算量大致相当于一个 520 亿参数的普通模型。内存由总参数量决定，速度主要由激活参数量决定。

---

(a) Attention is $512^2 + 2 \times 512 \times 128 + 512^2 = 655{,}360$. SwiGLU is $3 \times 512 \times 1024 = 1{,}572{,}864$. The norms are $1024$. One block has $2{,}229{,}248$ and 4 blocks have $8{,}916{,}992$. With the embedding $512{,}000$ and the final norm $512$ the total is $9{,}429{,}504$.

(b) At 16 bits $8.03 \times 10^9 \times 2 = 16.1$ GB. At 8 bits $8.0$ GB. At 4.89 bits $8.03 \times 10^9 \times 4.89 / 8 \approx 4.9$ GB. This agrees with the size of the Q4_K_M version of Llama-3.1-8B in the llama.cpp documentation.

(c) $q_{\max} = 3$ and $s = 1.4 / 3 \approx 0.467$. $w / s = (1.29,\ -0.43,\ 0.11,\ -3)$ rounds to $(1,\ 0,\ 0,\ -3)$, restored as $(0.467,\ 0,\ 0,\ -1.4)$. $-0.2$ and $0.05$ are both smaller in magnitude than $s/2 \approx 0.233$ and round to 0. The largest weight is kept exactly and the smaller weights lose the most.

(d) $10^{12}$ parameters at 1 byte each are $10^{12}$ bytes, that is 1000 GB. The computation per token corresponds roughly to an ordinary model with 52 billion parameters. Memory is determined by the total and speed mainly by the active parameters.

</details>